In [ ]:
!pip install pydantic-ai openai annotated-types -q

import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
print("✅ Готово")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.2/101.2 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.7/751.7 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.5/91.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.5/350.5 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from dataclasses import dataclass
from pydantic_ai import Agent, RunContext
from pydantic_ai.exceptions import ModelRetry
from pydantic import BaseModel, Field
from typing import Literal, Annotated
from annotated_types import Ge, Le

class TicketClassification(BaseModel):
    category: Literal["billing", "technical", "account", "other"]
    priority: Literal["low", "medium", "high", "critical"]
    summary: str = Field(max_length=100)
    requires_human: bool
    estimated_minutes: Annotated[int, Ge(1), Le(480)]

@dataclass
class TicketDeps:
    user_id: str
    role: Literal["agent", "supervisor"]
    max_priority: Literal["low", "medium", "high", "critical"]

PRIORITY_RANK = {"low": 0, "medium": 1, "high": 2, "critical": 3}
print("✅ Схемы готовы")

✅ Схемы готовы


In [ ]:
validation_log = []  # лог всех срабатываний валидатора

agent = Agent(
    'openai:gpt-4o-mini',
    output_type=TicketClassification,
    deps_type=TicketDeps,
    retries=3,  # ← макс. попыток при ModelRetry
    system_prompt="""Ты классифицируешь тикеты поддержки.
Если тикет требует эскалации — вызови инструмент escalate_ticket.

КРИТЕРИИ ПРИОРИТЕТА:
- critical: взлом аккаунта, полный сбой API, финансовые потери прямо сейчас
- high: двойное списание, невозможно войти, API 500, интеграция сломана
- medium: частичные ошибки, задержки, нужна помощь с настройкой
- low: вопросы как-сделать, запросы на изменение тарифа, пожелания

ВАЖНО:
- Если priority=critical, то requires_human ОБЯЗАТЕЛЬНО должно быть true
- estimated_minutes должно быть кратно 15 (15, 30, 45, 60, 90, 120...)"""
)

@agent.tool
async def escalate_ticket(
    ctx: RunContext[TicketDeps],
    ticket_id: str,
    new_priority: Literal["low", "medium", "high", "critical"]
) -> str:
    max_rank = PRIORITY_RANK[ctx.deps.max_priority]
    if PRIORITY_RANK[new_priority] > max_rank:
        raise ModelRetry(
            f"Role '{ctx.deps.role}' не может установить priority '{new_priority}'. "
            f"Максимально допустимый: '{ctx.deps.max_priority}'."
        )
    return f"✅ Ticket {ticket_id} escalated to '{new_priority}' by {ctx.deps.role}"

@agent.output_validator
async def validate_output(
    ctx: RunContext[TicketDeps],
    output: TicketClassification
) -> TicketClassification:

    errors = []

    # Проверка 1: critical → requires_human обязательно true
    if output.priority == "critical" and not output.requires_human:
        msg = (f"❌ Проверка 1 FAILED: priority='critical' но requires_human=False. "
               f"Critical тикеты ОБЯЗАНЫ иметь requires_human=True.")
        validation_log.append({"check": 1, "status": "RETRY", "msg": msg})
        print(f"  🔄 {msg}")
        errors.append(msg)

    # Проверка 2: estimated_minutes кратно 15
    if output.estimated_minutes % 15 != 0:
        rounded = round(output.estimated_minutes / 15) * 15
        rounded = max(15, min(480, rounded))  # держим в допустимом диапазоне
        msg = (f"❌ Проверка 2 FAILED: estimated_minutes={output.estimated_minutes} "
               f"не кратно 15. Используй {rounded}.")
        validation_log.append({"check": 2, "status": "RETRY", "msg": msg})
        print(f"  🔄 {msg}")
        errors.append(msg)

    if errors:
        raise ModelRetry("\n".join(errors))

    # Всё ок
    validation_log.append({
        "check": "all",
        "status": "PASS",
        "priority": output.priority,
        "requires_human": output.requires_human,
        "estimated_minutes": output.estimated_minutes
    })
    print(f"  ✅ Валидация пройдена: priority={output.priority}, "
          f"requires_human={output.requires_human}, "
          f"estimated_minutes={output.estimated_minutes}")
    return output

print("✅ Агент с output_validator готов")

✅ Агент с output_validator готов


In [ ]:
validation_log.clear()

ticket_critical = "Мой аккаунт был взломан, вижу подозрительные входы из другой страны"

deps = TicketDeps(
    user_id="supervisor_01",
    role="supervisor",
    max_priority="critical"
)

print("=" * 60)
print(f"📋 Тикет: {ticket_critical}")
print(f"👤 Роль: {deps.role} | max: {deps.max_priority}")
print("=" * 60)

result = await agent.run(ticket_critical, deps=deps)
r = result.output

print(f"\n📦 Финальный результат:")
print(f"   priority         : {r.priority}")
print(f"   requires_human   : {r.requires_human}")
print(f"   estimated_minutes: {r.estimated_minutes}")
print(f"   category         : {r.category}")
print(f"   summary          : {r.summary}")
print(f"\n📊 Retries использовано: {result.usage().requests - 1}")

📋 Тикет: Мой аккаунт был взломан, вижу подозрительные входы из другой страны
👤 Роль: supervisor | max: critical
  ✅ Валидация пройдена: priority=critical, requires_human=True, estimated_minutes=60

📦 Финальный результат:
   priority         : critical
   requires_human   : True
   estimated_minutes: 60
   category         : account
   summary          : Взлом аккаунта с подозрительными входами

📊 Retries использовано: 1


In [ ]:
import pandas as pd

test_tickets = [
    ("Взлом аккаунта, чужие входы из другой страны",    "supervisor", "critical"),
    ("API возвращает 500 при запросе с токеном",         "supervisor", "critical"),
    ("Не могу войти в аккаунт — неверный пароль",        "agent",      "high"),
    ("Как изменить валюту выставления счетов?",           "agent",      "high"),
    ("Приложение медленно загружается",                   "agent",      "high"),
]

validation_log.clear()
rows = []

for ticket_text, role, max_p in test_tickets:
    deps = TicketDeps(user_id=f"user_{role}", role=role, max_priority=max_p)
    print(f"\n▶ {ticket_text[:45]}...")
    res = await agent.run(ticket_text, deps=deps)
    r = res.output

    # Финальная проверка инвариантов
    assert not (r.priority == "critical" and not r.requires_human), \
        "ИНВАРИАНТ НАРУШЕН: critical без requires_human!"
    assert r.estimated_minutes % 15 == 0, \
        f"ИНВАРИАНТ НАРУШЕН: {r.estimated_minutes} не кратно 15!"

    rows.append({
        "ticket":           ticket_text[:40] + "...",
        "priority":         r.priority,
        "requires_human":   r.requires_human,
        "est_minutes":      r.estimated_minutes,
        "retries":          result.usage().requests - 1,
        "invariants_ok":    "✅"
    })

df = pd.DataFrame(rows)
print("\n\n📊 ИТОГОВАЯ ТАБЛИЦА:")
print(df.to_string(index=False))


▶ Взлом аккаунта, чужие входы из другой страны...
  ✅ Валидация пройдена: priority=critical, requires_human=True, estimated_minutes=30

▶ API возвращает 500 при запросе с токеном...
  ✅ Валидация пройдена: priority=high, requires_human=True, estimated_minutes=30

▶ Не могу войти в аккаунт — неверный пароль...
  ✅ Валидация пройдена: priority=high, requires_human=True, estimated_minutes=30

▶ Как изменить валюту выставления счетов?...
  ✅ Валидация пройдена: priority=low, requires_human=False, estimated_minutes=15

▶ Приложение медленно загружается...
  ✅ Валидация пройдена: priority=medium, requires_human=False, estimated_minutes=15


📊 ИТОГОВАЯ ТАБЛИЦА:
                                     ticket priority  requires_human  est_minutes  retries invariants_ok
Взлом аккаунта, чужие входы из другой ст... critical            True           30        1             ✅
API возвращает 500 при запросе с токеном...     high            True           30        1             ✅
Не могу войти в аккау

In [ ]:
from google.colab import auth
import gspread
from google.auth import default
import pandas as pd
from datetime import datetime

# --- Авторизация Google ---
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SHEET_ID = "1b-zF3JJAq5C7GFhn2N3pJczD829lWOho0RoeSoKqdlQ"
spreadsheet = gc.open_by_key(SHEET_ID)

# --- Собираем финальный отчёт ---
report_rows = []

for ticket_text, role, max_p in test_tickets:
    deps = TicketDeps(user_id=f"user_{role}", role=role, max_priority=max_p)
    res = await agent.run(ticket_text, deps=deps)
    r = res.output

    retry_count = res.usage().requests - 1
    invariant_ok = (
        not (r.priority == "critical" and not r.requires_human)
        and r.estimated_minutes % 15 == 0
    )

    report_rows.append({
        "timestamp":         datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "ticket":            ticket_text,
        "role":              role,
        "max_priority":      max_p,
        "category":          r.category,
        "priority":          r.priority,
        "summary":           r.summary,
        "requires_human":    str(r.requires_human),
        "estimated_minutes": r.estimated_minutes,
        "retries":           retry_count,
        "minutes_div15":     r.estimated_minutes % 15 == 0,
        "invariants_ok":     invariant_ok,
    })

    status = "✅" if invariant_ok else "❌"
    print(f"{status} [{role}] {ticket_text[:45]}... "
          f"→ {r.priority} | human={r.requires_human} | "
          f"{r.estimated_minutes}min | retries={retry_count}")

report_df = pd.DataFrame(report_rows)

# --- Итоговая статистика в консоль ---
print("\n" + "=" * 60)
print("📊 ИТОГОВЫЙ ОТЧЁТ:")
print(f"   Всего тикетов      : {len(report_df)}")
print(f"   Инварианты OK      : {report_df['invariants_ok'].sum()}/{len(report_df)}")
print(f"   Critical тикетов   : {(report_df['priority'] == 'critical').sum()}")
print(f"   Requires human     : {(report_df['requires_human'] == 'True').sum()}")
print(f"   Всего retries      : {report_df['retries'].sum()}")
print(f"   Minutes кратно 15  : {report_df['minutes_div15'].sum()}/{len(report_df)}")
print("=" * 60)

# --- Запись в новый лист Google Sheets ---
sheet_name = f"validation_{datetime.now().strftime('%m%d_%H%M')}"

try:
    ws = spreadsheet.worksheet(sheet_name)
    ws.clear()
except gspread.exceptions.WorksheetNotFound:
    ws = spreadsheet.add_worksheet(title=sheet_name, rows=50, cols=20)

# Заголовки + данные
header = report_df.columns.tolist()
data   = report_df.astype(str).values.tolist()
ws.update([header] + data)

# Форматирование заголовка — жирный + заморозить первую строку
ws.format("A1:L1", {
    "textFormat": {"bold": True},
    "backgroundColor": {"red": 0.2, "green": 0.6, "blue": 0.8}
})
ws.freeze(rows=1)

print(f"\n✅ Записано {len(report_df)} строк на лист '{sheet_name}'")
print(f"🔗 https://docs.google.com/spreadsheets/d/{SHEET_ID}/edit")

  ✅ Валидация пройдена: priority=critical, requires_human=True, estimated_minutes=15
✅ [supervisor] Взлом аккаунта, чужие входы из другой страны... → critical | human=True | 15min | retries=1
  ✅ Валидация пройдена: priority=high, requires_human=True, estimated_minutes=30
✅ [supervisor] API возвращает 500 при запросе с токеном... → high | human=True | 30min | retries=1
  ✅ Валидация пройдена: priority=high, requires_human=True, estimated_minutes=15
✅ [agent] Не могу войти в аккаунт — неверный пароль... → high | human=True | 15min | retries=0
  ✅ Валидация пройдена: priority=low, requires_human=False, estimated_minutes=15
✅ [agent] Как изменить валюту выставления счетов?... → low | human=False | 15min | retries=0
  ✅ Валидация пройдена: priority=medium, requires_human=False, estimated_minutes=30
✅ [agent] Приложение медленно загружается... → medium | human=False | 30min | retries=0

📊 ИТОГОВЫЙ ОТЧЁТ:
   Всего тикетов      : 5
   Инварианты OK      : 5/5
   Critical тикетов   : 1
   Requ